# Notebook 22 — Augmented TrOCR + External Formulary (the combined best model)

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

DATA_ROOT = Path("../data/pharmacy_lk")
TRAIN_CSV = DATA_ROOT/"splits/train.csv"; TEST_CSV = DATA_ROOT/"splits/test.csv"
LABEL_COL = "medicine_name"; IMG_COL = "image_filename"
AUG_CKPT = Path("../checkpoints/trocr_augmented/best")     # the augmented model
PROC = "microsoft/trocr-small-handwritten"
FORMULARY_CSV = Path("../data/formulary/drug_names.csv")

Using device: mps


## 1. Metrics + lexicon machinery

In [2]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur=[i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]
def metrics(preds, refs):
    tot=sum(edit_distance(p,r) for p,r in zip(preds,refs)); chars=sum(len(r) for r in refs)
    em=sum(p==r for p,r in zip(preds,refs))
    return {"CER":tot/max(chars,1),"EM":em/len(refs),"n":len(refs)}

train_names = sorted(set(pd.read_csv(TRAIN_CSV)[LABEL_COL].astype(str).str.strip().str.lower()))
if FORMULARY_CSV.exists():
    ext = pd.read_csv(FORMULARY_CSV); col = "drug_name" if "drug_name" in ext.columns else ext.columns[0]
    external = sorted(set(ext[col].astype(str).str.strip().str.lower()) - {""} | set(train_names))
    real = True
else:
    test_names = sorted(set(pd.read_csv(TEST_CSV)[LABEL_COL].astype(str).str.strip().str.lower()))
    external = sorted(set(train_names) | set(test_names)); real = False
    print("!! No formulary found — DEMO mode (peeks at test, upper-bound only).")
print(f"training lexicon: {len(train_names)} | external formulary: {len(external)} (real={real})")

def build_index(names):
    bl = defaultdict(list)
    for w in names: bl[len(w)].append(w)
    return set(names), bl
train_set, train_idx = build_index(train_names)
ext_set, ext_idx = build_index(external)

def nearest(word, by_len, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd
def snap(word, by_len, tau=0.4):
    e, d = nearest(word, by_len)
    return e if (e is not None and d/max(len(word),1) <= tau) else word

training lexicon: 1210 | external formulary: 4779 (real=True)


## 2. Augmented TrOCR raw predictions on test

In [3]:
processor = TrOCRProcessor.from_pretrained(PROC)
trocr = VisionEncoderDecoderModel.from_pretrained(AUG_CKPT).to(DEVICE).eval()

def trocr_one(pil):
    pv = processor(pil.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        ids = trocr.generate(pv, max_new_tokens=24)
    return processor.decode(ids[0], skip_special_tokens=True).strip().lower()

df = pd.read_csv(TEST_CSV).dropna(subset=[LABEL_COL])
raw, refs, files = [], [], []
print("running AUGMENTED TrOCR on test...")
for _, r in df.iterrows():
    fn=str(r[IMG_COL]); p=DATA_ROOT/"images"/fn
    if not p.exists(): continue
    raw.append(trocr_one(Image.open(p)))
    refs.append(str(r[LABEL_COL]).strip().lower()); files.append(fn)
print(f"{len(raw)} predictions")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

running AUGMENTED TrOCR on test...
791 predictions


## 3. Compare combinations

In [4]:
pred_train = [snap(p, train_idx) for p in raw]
pred_ext   = [snap(p, ext_idx)   for p in raw]

print("TEST — AUGMENTED TrOCR combinations:")
for name, preds in [("Aug TrOCR raw", raw),
                    ("Aug TrOCR + training lexicon", pred_train),
                    ("Aug TrOCR + external formulary", pred_ext)]:
    m = metrics(preds, refs)
    print(f"  {name:34s}: CER {m['CER']:.4f} | EM {m['EM']:.4f}")

seen_mask = [r in train_set for r in refs]
def split_em(preds):
    s=[(p,r) for p,r,se in zip(preds,refs,seen_mask) if se]
    u=[(p,r) for p,r,se in zip(preds,refs,seen_mask) if not se]
    se=metrics([p for p,_ in s],[r for _,r in s])["EM"] if s else 0
    ue=metrics([p for p,_ in u],[r for _,r in u])["EM"] if u else 0
    return se,ue,len(s),len(u)
print("\nseen / unseen EM:")
for name, preds in [("Aug raw",raw),("+ training lexicon",pred_train),("+ external formulary",pred_ext)]:
    se,ue,ns,nu = split_em(preds)
    print(f"  {name:24s}: seen {se:.3f} (n={ns}) | unseen {ue:.3f} (n={nu})")

unseen_names = sorted({r for r,se in zip(refs,seen_mask) if not se})
covered = sum(1 for n in unseen_names if n in ext_set)
print(f"\nformulary covers {covered}/{len(unseen_names)} unseen names ({covered/max(len(unseen_names),1):.1%})")

pd.DataFrame([{"model":"AugTrOCR", **metrics(raw,refs)},
              {"model":"AugTrOCR+trainlex", **metrics(pred_train,refs)},
              {"model":"AugTrOCR+formulary", **metrics(pred_ext,refs)}]
            ).to_csv("../checkpoints/trocr_augmented/combined_results.csv", index=False)

TEST — AUGMENTED TrOCR combinations:
  Aug TrOCR raw                     : CER 0.1762 | EM 0.6549
  Aug TrOCR + training lexicon      : CER 0.1887 | EM 0.6612
  Aug TrOCR + external formulary    : CER 0.1668 | EM 0.7206

seen / unseen EM:
  Aug raw                 : seen 0.810 (n=615) | unseen 0.114 (n=176)
  + training lexicon      : seen 0.844 (n=615) | unseen 0.023 (n=176)
  + external formulary    : seen 0.841 (n=615) | unseen 0.301 (n=176)

formulary covers 156/172 unseen names (90.7%)


## 4. Full progression (for the thesis) — every validated step

In [5]:
print("""
FULL PROGRESSION (custom test, overall EM):
  TrOCR (no aug)                       0.569
  TrOCR + training lexicon             0.612
  TrOCR + external formulary           0.659
  Augmented TrOCR                      0.655
  Augmented TrOCR + external formulary (above) <- combined best

Two complementary, independently-validated improvements:
  (1) augmentation -> better core recognition (lower CER)
  (2) external formulary -> recovers unseen names (open-vocabulary gap)
Report both honestly; the combined model is the culmination.
""")


FULL PROGRESSION (custom test, overall EM):
  TrOCR (no aug)                       0.569
  TrOCR + training lexicon             0.612
  TrOCR + external formulary           0.659
  Augmented TrOCR                      0.655
  Augmented TrOCR + external formulary (above) <- combined best

Two complementary, independently-validated improvements:
  (1) augmentation -> better core recognition (lower CER)
  (2) external formulary -> recovers unseen names (open-vocabulary gap)
Report both honestly; the combined model is the culmination.

